## 面试问题

无进展 loop stall 怎么检测重复动作/震荡并打破？

## 回答主线

循环可能重复动作、状态震荡或语义打转。只靠 max_steps 会白烧预算。做法是独立的 stall 检测：记录最近(动作,状态签名)，检测连续无进展并主动打破。本 Notebook 用一个反复查同一不存在订单的 agent，对比只靠 max_steps(烧满8步)与 stall 检测(第3次重复即停)，并用一个「动作相同但每步读不同页」的有进展 agent 证明不会误判。

## 真实案例

坏 agent 每步都 `lookup` 同一不存在订单，状态签名恒为 `order-missing`（无进展）。对比只靠步数上限与 stall 检测；再用分页 agent（签名每步变化）验证有进展的重复不被误判。数据为教学循环，不代表真实 agent。

In [1]:
def stubborn_agent(state):  # 一个反复查询同一不存在订单的坏 agent。
    return {"action": "lookup", "signature": state["signature"]}  # 动作与状态签名都不变。

initial = {"signature": "order-missing"}  # 初始状态签名恒不变代表无进展。
print("坏 agent 每步动作:", stubborn_agent(initial)["action"])  # 展示每步都是 lookup。
print("坏 agent 状态签名:", initial["signature"])  # 展示签名不变。
print("该 agent 永不改变状态签名, 即无进展")  # 说明无进展本质。

坏 agent 每步动作: lookup
坏 agent 状态签名: order-missing
该 agent 永不改变状态签名, 即无进展


## 基线（Baseline）

反面基线：只靠 `max_steps`。坏 agent 会一直重复到烧满 8 步才停，白白浪费全部预算，且无法区分「在打转」和「在推进」。

In [2]:
def run_only_maxsteps(agent, state, max_steps=8):  # 只靠步数上限的循环。
    steps = 0  # 记录步数。
    for _ in range(max_steps):  # 最多执行 max_steps 步。
        agent(state)  # 执行一步（无进展）。
        steps += 1  # 累加步数。
    return {"stop_reason": "max_steps", "steps": steps}  # 烧满步数才停。

only_max = run_only_maxsteps(stubborn_agent, initial)  # 运行只靠步数上限的循环。
print("只靠 max_steps 结果:", only_max)  # 展示白白烧满 8 步。

只靠 max_steps 结果: {'stop_reason': 'max_steps', 'steps': 8}


## 失败案例与修正

只靠步数上限延迟暴露无进展。修正是 stall 检测：记录最近(动作,签名)，当同一 key 连续达到阈值就判定 stall 并打破。有进展的重复（签名每步变化）不会触发。

In [3]:
def run_with_stall_detection(agent, state, max_steps=8, stall_threshold=3):  # 带 stall 检测的循环。
    recent = []  # 记录最近的动作与签名。
    steps = 0  # 记录步数。
    for _ in range(max_steps):  # 最多执行 max_steps 步。
        proposal = agent(state)  # agent 提议动作。
        key = (proposal["action"], proposal["signature"])  # 组合动作与状态签名。
        recent.append(key)  # 记入最近历史。
        steps += 1  # 累加步数。
        if recent[-stall_threshold:].count(key) == stall_threshold:  # 同一 key 连续达到阈值。
            return {"stop_reason": "stall_detected", "steps": steps}  # 检测到无进展提前打破。
    return {"stop_reason": "max_steps", "steps": steps}  # 未检测到则烧满步数。

In [4]:
with_stall = run_with_stall_detection(stubborn_agent, initial)  # 运行带 stall 检测的循环。
print("带 stall 检测结果:", with_stall)  # 展示第 3 步就打破。
print("坏 agent 节省步数:", only_max["steps"] - with_stall["steps"])  # 展示节省的步数。

带 stall 检测结果: {'stop_reason': 'stall_detected', 'steps': 3}
坏 agent 节省步数: 5


In [5]:
def paging_agent(state):  # 一个每步读取不同页的有进展 agent。
    page = state["page"]  # 取当前页。
    state["page"] = page + 1  # 推进到下一页（有进展）。
    return {"action": "lookup", "signature": "page-" + str(page)}  # 动作相同但签名每步不同。

paging_state = {"page": 0}  # 初始化分页状态。
prog = run_with_stall_detection(paging_agent, paging_state, max_steps=4)  # 对有进展 agent 运行同一检测。
print("有进展 agent 结果:", prog)  # 展示动作相同但签名变化不被误判为 stall。
print("只靠 max_steps 步数:", only_max["steps"], "| stall 检测步数:", with_stall["steps"])  # 对比步数。

有进展 agent 结果: {'stop_reason': 'max_steps', 'steps': 4}
只靠 max_steps 步数: 8 | stall 检测步数: 3


## 结果解读

坏 agent 在只靠 max_steps 时烧满 8 步，stall 检测第 3 次重复就打破、省下 5 步；分页 agent 动作相同但状态签名每步变化，不触发 stall，跑满计划步数。要点：签名反映实质进展，区分「重复有进展」与「重复无进展」，打破优先换策略或升级。

In [6]:
assert only_max["steps"] == 8  # 只靠 max_steps 烧满 8 步。
assert with_stall["stop_reason"] == "stall_detected"  # stall 检测提前打破。
assert with_stall["steps"] == 3  # 连续 3 次无进展即打破。
assert with_stall["steps"] < only_max["steps"]  # stall 检测节省了步数。
assert prog["stop_reason"] == "max_steps"  # 有进展的重复不被误判为 stall。
print("全部不变量通过")  # 输出测试通过信号。

全部不变量通过
